# Prototipo de la métrica Macro AP-rIoU

Este notebook es el espacio de experimentación de la issue #9. Saúl y Dolly lo usarán para comprender y comprobar manualmente cada parte de la métrica antes de trasladarla a `src/evaluation/metric.py`.

**Estado actual:** primer experimento de representación de una OBB. Todavía no se calcula rIoU, matching ni AP.

## 1. Contrato que debemos respetar

Una OBB se representa como `(cx, cy, width, height, angle_deg)`. El centro `(cx, cy)`, el ancho y el alto se expresan en píxeles; el ángulo se expresa en grados.

Una predicción completa usa `(frame_id, score, cx, cy, width, height, angle_deg)`. El ground truth no contiene `score` porque es la respuesta correcta y no una estimación del modelo.

En este primer experimento la entrada es una OBB paramétrica y la salida esperada son sus cuatro vértices en píxeles.

In [ ]:
import math

import cv2
import numpy as np
from IPython.display import Image, display

## 2. De parámetros a cuatro vértices

Primero colocamos cuatro esquinas alrededor del origen: `(-w/2, -h/2)`, `(w/2, -h/2)`, `(w/2, h/2)` y `(-w/2, h/2)`. Después las rotamos por `angle_deg` y finalmente trasladamos todas al centro `(cx, cy)`.

La función siguiente es deliberadamente experimental. Cuando entendamos y validemos todos sus casos, la implementación definitiva se escribirá y probará en `metric.py`.

In [ ]:
def experimental_obb_to_vertices(obb):
    cx, cy, width, height, angle_deg = obb
    theta = math.radians(angle_deg)
    rotation = np.array(
        [
            [math.cos(theta), -math.sin(theta)],
            [math.sin(theta), math.cos(theta)],
        ],
        dtype=np.float64,
    )
    local_vertices = np.array(
        [
            [-width / 2, -height / 2],
            [width / 2, -height / 2],
            [width / 2, height / 2],
            [-width / 2, height / 2],
        ],
        dtype=np.float64,
    )
    return local_vertices @ rotation.T + np.array([cx, cy])


axis_aligned_obb = (200.0, 150.0, 120.0, 60.0, 0.0)
axis_aligned_vertices = experimental_obb_to_vertices(axis_aligned_obb)
expected_vertices = np.array(
    [[140.0, 120.0], [260.0, 120.0], [260.0, 180.0], [140.0, 180.0]]
)

assert np.allclose(axis_aligned_vertices, expected_vertices)
assert math.isclose(cv2.contourArea(axis_aligned_vertices.astype(np.float32)), 120.0 * 60.0)

print("Vértices calculados:")
print(axis_aligned_vertices)
print()
print("Comprobación: área del polígono = width × height = 7200 px²")

## 3. Visualización de una OBB rotada

Ahora conservamos el mismo centro, ancho y alto, pero usamos un ángulo de 30°. La rotación debe cambiar los vértices sin cambiar el centro ni el área.

In [ ]:
rotated_obb = (200.0, 150.0, 120.0, 60.0, 30.0)
rotated_vertices = experimental_obb_to_vertices(rotated_obb)
rotated_area = cv2.contourArea(rotated_vertices.astype(np.float32))

assert math.isclose(rotated_area, 120.0 * 60.0, rel_tol=1e-6)
assert np.allclose(rotated_vertices.mean(axis=0), [200.0, 150.0])

canvas = np.full((300, 400, 3), 245, dtype=np.uint8)
polygon = np.rint(rotated_vertices).astype(np.int32).reshape((-1, 1, 2))
cv2.polylines(canvas, [polygon], isClosed=True, color=(40, 120, 220), thickness=3)
cv2.circle(canvas, (200, 150), radius=5, color=(220, 60, 40), thickness=-1)
cv2.putText(
    canvas,
    "centro (200, 150)",
    (210, 145),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.5,
    (40, 40, 40),
    1,
    cv2.LINE_AA,
)
success, encoded_image = cv2.imencode(".png", canvas)
assert success

print(f"Centro recuperado: {rotated_vertices.mean(axis=0)}")
print(f"Área después de rotar: {rotated_area:.1f} px²")
display(Image(data=encoded_image.tobytes()))

## 4. Qué debemos entender antes de continuar

1. `cx` y `cy` indican el centro, no una esquina.
2. Antes de rotar, las esquinas se construyen usando la mitad del ancho y del alto.
3. La rotación cambia la posición de los vértices, pero conserva el centro y el área.
4. Los vértices son necesarios porque rIoU compara la intersección de dos polígonos rotados.

**Siguiente experimento pendiente:** validar dimensiones y ángulos problemáticos antes de calcular la intersección entre dos OBB.